# Maritime traffic analytics: exploration notebook

**Question:** how busy and how congested is shipping in the North Sea and English Channel, how good is the AIS data, and what does waiting cost?

**Before running:** `python analysis/run_all.py` (it cleans the data and writes `outputs/data/`). Then *Run All* here.
Cells marked **Your interpretation** are for you to fill in: a number without a sentence on what it means is only half an analysis.

In [ ]:
import sys, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
DATA = ROOT / "outputs" / "data"
pd.set_option("display.max_columns", 40, "display.width", 150)
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 100})

ships = pd.read_csv(DATA / "ships_clean.csv")
hub_table = pd.read_csv(DATA / "hub_table.csv")
log = pd.read_csv(DATA / "cleaning_log.csv")
print(f"{len(ships):,} current ships (heard from in the last 15 minutes), {len(hub_table)} port hubs")

## 1. Data quality audit
Before any analysis: how much of the data can be trusted? `run_all.py` applies explicit cleaning rules and counts every row it touches.

In [ ]:
display(log)
completeness = pd.read_csv(DATA / "completeness.csv")
display(completeness.sort_values("pct_complete"))

**Your interpretation:** which fields are least complete, and which analyses does that rule out or weaken? (For example, ship length is needed to scale fuel use by size.)

### Destination text: the messiest field
Captains type this by hand. Compare the raw text with what cleaning made of it.

In [ ]:
sample = (ships.dropna(subset=["destination"])
          .drop_duplicates("destination")[["destination", "dest_clean", "dest_quality", "dest_hub", "destination_port"]]
          .head(25))
display(sample)
print(ships["dest_quality"].value_counts(normalize=True).mul(100).round(1).to_string())

## 2. Fleet composition

In [ ]:
mix = ships.groupby("vessel_type").agg(ships=("mmsi", "size"), pct_moving=("status_group", lambda s: round(100 * (s == "moving").mean(), 1)))
mix["share_%"] = (100 * mix["ships"] / mix["ships"].sum()).round(1)
mix = mix.sort_values("ships", ascending=False)
display(mix)
display(pd.pivot_table(ships, index="vessel_type", columns="status_group", values="mmsi", aggfunc="count", fill_value=0))
mix["ships"].plot.bar(figsize=(8, 3.5), color="#1d8f87", title="Ships by vessel type")
plt.show()

**Your interpretation:** how large is the 'unknown' type group, and what does that do to the shares above?

## 3. Congestion by port hub
Congestion here means ships **waiting at anchor** inside a hub's radius. Levels are relative: the lowest third of hubs is green, the middle third yellow, the top third red.

In [ ]:
t = hub_table.sort_values("at_anchor")
colors = t["level_waiting"].map({"green": "#2e9e5b", "yellow": "#e0b020", "red": "#d64541"})
ax = t.plot.barh(x="hub", y="at_anchor", color=colors, legend=False, figsize=(8, 6), title="Ships waiting at anchor near each hub")
ax.set_xlabel("ships at anchor")
plt.show()
display(hub_table[["hub", "zone", "ships_in_radius", "at_anchor", "pct_at_anchor", "avg_speed_moving_kn", "level_waiting"]].head(10))

**Your interpretation:** which hubs stand out, and is that because they are big (many ships) or because a large *share* of their ships is waiting? Compare `ships_in_radius` with `pct_at_anchor`.

## 4. Do ships slow down near hubs?
A difference in average speed is only worth reporting with a sense of its uncertainty, so this uses a bootstrap confidence interval (no extra libraries needed).

In [ ]:
mv = ships[(ships["status_group"] == "moving") & ships["speed"].notna()].copy()
mv["near_hub"] = mv["hub"].notna()
near, open_ = mv.loc[mv["near_hub"], "speed"].to_numpy(), mv.loc[~mv["near_hub"], "speed"].to_numpy()
print(f"moving ships near a hub: {len(near):,} (mean {near.mean():.2f} kn) | in open water: {len(open_):,} (mean {open_.mean():.2f} kn)" if len(near) and len(open_) else "not enough moving ships")
if len(near) >= 10 and len(open_) >= 10:
    rng = np.random.default_rng(1)
    diffs = [rng.choice(near, len(near)).mean() - rng.choice(open_, len(open_)).mean() for _ in range(2000)]
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    print(f"difference (near - open): {near.mean() - open_.mean():+.2f} kn, 95% bootstrap interval [{lo:+.2f}, {hi:+.2f}]")
    print("The interval excludes 0: a real difference." if (lo > 0 or hi < 0) else "The interval includes 0: no clear difference.")
    mv.boxplot(column="speed", by="near_hub", figsize=(6, 4)); plt.suptitle(""); plt.title("Speed of moving ships: near a hub (True) vs open water (False)"); plt.show()

**Your interpretation:** what could explain the result besides congestion (pilotage rules, speed limits in harbour approaches, ship types that differ between areas)?

## 5. Silent-ship flags: coverage or behaviour?
If flags are concentrated at the edge of the tracked area, many are probably ships that sailed out of receiver coverage. The comparison uses only ships heard in the last hour, so the two areas are treated equally.

In [ ]:
ships_all = pd.read_csv(DATA / "ships_all.csv")   # includes ships that have gone silent: the flag only exists for them
# equal-age window: the backend keeps silent ships longer in the interior than near the edge, so compare only ships heard in the last hour
ships_all = ships_all[ships_all["minutes_since_update"].fillna(0) <= 60]
g = ships_all.groupby("is_edge")["is_dark_flagged"].agg(["sum", "count"])
g.index = g.index.map({0: "interior", 1: "near edge"})
g["rate_%"] = (100 * g["sum"] / g["count"]).round(2)
display(g)
if {"interior", "near edge"} <= set(g.index):
    p1, n1, p2, n2 = g.loc["near edge", "sum"] / g.loc["near edge", "count"], g.loc["near edge", "count"], g.loc["interior", "sum"] / g.loc["interior", "count"], g.loc["interior", "count"]
    se = np.sqrt(p1 * (1 - p1) / n1 + p2 * (1 - p2) / n2)
    print(f"edge minus interior: {100 * (p1 - p2):+.2f} percentage points (approx. 95% CI +/- {196 * se:.2f})")

**Your interpretation:** does the detector measure suspicious behaviour, or the limits of the receiver network?

## 6. Voyages

In [ ]:
routes = pd.read_csv(DATA / "routes.csv")
display(routes)
print(f"{ships['departure_port'].notna().sum():,} ships have a detected departure port")

## 7. Cost of waiting: estimate and sensitivity
These are **estimates** from published ranges (`config/assumptions.csv`). The sensitivity table shows which assumption matters most, which is the honest way to present an estimate.

In [ ]:
from ais import config, estimates
assump = estimates.load_assumptions(config.ASSUMPTIONS_CSV)
est = estimates.estimate_waiting(ships, assump)
display(est.round(1))

w = ships[(ships["status_group"] == "at_anchor") & ships["hub"].notna()]
counts = w.groupby("size_class").size()

def total_cost(fuel="central", price="central", opex="central"):
    p, o = estimates.lookup(assump, "fuel_price_usd_per_t")[price], estimates.lookup(assump, "vessel_opex_usd_per_day")[opex]
    return sum(n * (o + estimates.lookup(assump, "fuel_at_anchor_t_per_day", sc)[fuel] * p) for sc, n in counts.items())

base = total_cost()
rows = []
for name, kw in (("fuel burn at anchor", "fuel"), ("fuel price", "price"), ("running cost (OPEX)", "opex")):
    lo, hi = total_cost(**{kw: "low"}), total_cost(**{kw: "high"})
    rows.append({"assumption varied": name, "cost low ($m/day)": round(lo / 1e6, 2), "central ($m/day)": round(base / 1e6, 2), "cost high ($m/day)": round(hi / 1e6, 2), "swing ($m)": round((hi - lo) / 1e6, 2)})
display(pd.DataFrame(rows).sort_values("swing ($m)", ascending=False))

**Your interpretation:** which assumption drives the result, and what data would you ask a shipping company for to replace it?

## 8. Time analysis (needs the history collector)
Once `collector.py` has run for a few days, this section fills in by itself.

In [ ]:
from ais import timeseries
hist = timeseries.load_history(config.HISTORY_DB)
if hist is None:
    print("No history yet. Start the collector:  caffeinate -i python analysis/collector.py")
else:
    tr = timeseries.analyse(hist, assump)
    print(tr["status"])
    display(tr["daily"].head(10))
    for msg in tr["messages"]:
        print("note:", msg)
    print("Calendar chart: outputs/time/t2_calendar.png (after run_all.py)")

## Key takeaways (write these yourself)
1. **Data quality:** ...
2. **Congestion:** ...
3. **Speed:** ...
4. **Dark fleet:** ...
5. **Cost of waiting:** ... (always say *estimate* and give the range)
6. **What I would do next, and what I could not conclude:** ...